# Notebook 04 - Price of Anarchy

This notebook compares the total system cost at the user-equilibrium assignment with the total system cost at the system-optimum assignment for the same finite route-choice instance.

For this numerical instance,

$$\mathrm{PoA}=\frac{\mathrm{TC}(x^{UE})}{\mathrm{TC}(x^{SO})},$$

where

$$\mathrm{TC}(x)=\sum_{e\in E}x_e t_e(x_e).$$

In exact arithmetic, $\mathrm{PoA}\ge 1$ because $x^{SO}$ minimizes $\mathrm{TC}$ over the feasible set. Values below $1$ indicate numerical error, inconsistent inputs, or a failed solve.

The theoretical bound reported below is a reference calculation for a class of polynomial latency functions. It is not estimated from Chapinero data.


In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import osmnx as ox
import networkx as nx
from pathlib import Path
from itertools import islice

from src.graph_utils import load_graph, bpr

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print('Libraries loaded.')

## 1. Load UE and SO results

The notebook uses saved results from Notebooks 02 and 03. The comparison is meaningful only if both previous solves were successful for the same instance.


In [ ]:
with open(PROCESSED_DIR / 'results_so.pkl', 'rb') as fh:
    res = pickle.load(fh)

x_ue       = res['x_ue']
x_so       = res['x_so']
t0         = res['t0']
cap        = res['cap']
cost_ue    = res['cost_ue']
cost_so    = res['cost_so']
poa        = res['poa']
edges_list = res['edges_list']
od_pairs   = res['od_pairs']
route_list = res['route_list']
od_route_map = res['od_route_map']
delta      = res['delta']

E = len(edges_list)
R = len(route_list)
total_demand = sum(dem for _, _, dem in od_pairs)

print(f'PoA base (demand actual): {poa:.6f}')
print(f'Costo UE: {cost_ue:,.0f} veh·s')
print(f'Costo SO: {cost_so:,.0f} veh·s')

In [ ]:
G = load_graph(RAW_DIR / 'chapinero_drive_enriched.graphml')

## 2. Compute the base PoA

The base value is computed from the saved UE and SO total system costs.


In [ ]:
def poa_upper_bound(beta=4):
    """
    Roughgarden upper bound for polynomial functions of degree beta.
    PoA <= (beta+1) * (beta/(beta+1))^beta  [simplified BPR version]
    """
    # Cota exacta: PoA <= 1 / (1 - alpha_p) donde alpha_p = p^p / (p+1)^(p+1)
    p = beta  # grado del polinomio de cost
    alpha_p = (p ** p) / ((p + 1) ** (p + 1))
    return 1 / (1 - alpha_p)

theoretical_bound = poa_upper_bound(beta=4)
ineficiencia_pct = (poa - 1) * 100
bound_pct = (theoretical_bound - 1) * 100

print('=' * 55)
print('   PRICE OF ANARCHY — Chapinero, Bogota')
print('=' * 55)
print(f'  PoA observado            : {poa:.6f}  (+{ineficiencia_pct:.3f}%)')
print(f'  Theoretical bound (BPR β=4)   : {theoretical_bound:.6f}  (+{bound_pct:.1f}%)')
print(f'  PoA as % of bound    : {poa/theoretical_bound*100:.1f}%')
print('-' * 55)
print(f'  Extra anarchy cost    : {cost_ue - cost_so:,.0f} veh·s')
print(f'  Tiempo perdido prom.     : {(cost_ue-cost_so)/total_demand:.1f} s/veh')
print('=' * 55)

## 3. Demand-scaling sensitivity

The demand vector is multiplied by scalar factors and the UE and SO problems are solved again. This is a numerical sensitivity exercise, not a demand forecast.


In [ ]:
def solve_ue(delta, t0, cap, od_pairs, od_route_map, demand_scale=1.0, alpha=0.15, beta=4):
    """Solve UE (Beckmann) for a given demand scale factor."""
    R = delta.shape[1]
    f = cp.Variable(R, nonneg=True)
    x = delta @ f

    term_linear = cp.sum(cp.multiply(t0, x))
    term_power  = cp.sum(
        cp.multiply(t0 * alpha / (beta + 1) / (cap ** beta), cp.power(x, beta + 1))
    )
    obj = cp.Minimize(term_linear + term_power)

    cons = []
    for (o, d, dem) in od_pairs:
        if (o, d) in od_route_map:
            cons.append(cp.sum(f[od_route_map[(o, d)]]) == dem * demand_scale)

    prob = cp.Problem(obj, cons)
    prob.solve(solver=cp.SCS, eps=1e-4, max_iters=10000, verbose=False)

    if prob.status not in ['optimal', 'optimal_inaccurate']:
        return None, None

    x_val = delta @ f.value
    t_val = t0 * (1 + alpha * (x_val / cap) ** beta)
    cost  = float(np.sum(t_val * x_val))
    return x_val, cost


def solve_so(delta, t0, cap, od_pairs, od_route_map, demand_scale=1.0, alpha=0.15, beta=4):
    """Solve SO (total-cost minimization) for a given scale factor."""
    R = delta.shape[1]
    f = cp.Variable(R, nonneg=True)
    x = delta @ f

    term_linear = cp.sum(cp.multiply(t0, x))
    term_power  = cp.sum(
        cp.multiply(t0 * alpha / (cap ** beta), cp.power(x, beta + 1))
    )
    obj = cp.Minimize(term_linear + term_power)

    cons = []
    for (o, d, dem) in od_pairs:
        if (o, d) in od_route_map:
            cons.append(cp.sum(f[od_route_map[(o, d)]]) == dem * demand_scale)

    prob = cp.Problem(obj, cons)
    prob.solve(solver=cp.SCS, eps=1e-4, max_iters=10000, verbose=False)

    if prob.status not in ['optimal', 'optimal_inaccurate']:
        return None, None

    x_val = delta @ f.value
    t_val = t0 * (1 + alpha * (x_val / cap) ** beta)
    cost  = float(np.sum(t_val * x_val))
    return x_val, cost


print('Solver functions defined.')

In [ ]:
# Escalar demand de 10% a 200%
scales = np.array([0.10, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00])
poa_curve = []
cost_ue_curve = []
cost_so_curve = []

for s in scales:
    _, c_ue = solve_ue(delta, t0, cap, od_pairs, od_route_map, demand_scale=s)
    _, c_so = solve_so(delta, t0, cap, od_pairs, od_route_map, demand_scale=s)
    if c_ue is not None and c_so is not None and c_so > 0:
        poa_s = c_ue / c_so
        poa_curve.append(poa_s)
        cost_ue_curve.append(c_ue)
        cost_so_curve.append(c_so)
        print(f'  Escala {s:.2f}x  |  UE={c_ue:,.0f}  SO={c_so:,.0f}  PoA={poa_s:.5f}')
    else:
        poa_curve.append(np.nan)
        cost_ue_curve.append(np.nan)
        cost_so_curve.append(np.nan)
        print(f'  Escala {s:.2f}x  |  no solution')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: curva de PoA
ax = axes[0]
ax.plot(scales * total_demand, poa_curve, 'o-', color='#e74c3c', linewidth=2, markersize=7)
ax.axhline(theoretical_bound, color='gray', linestyle='--', linewidth=1.2,
           label=f'Roughgarden bound ({theoretical_bound:.3f})')
ax.axhline(1.0, color='black', linestyle=':', linewidth=1)
ax.set_xlabel('Demand total (veh/h)', fontsize=11)
ax.set_ylabel('Price of Anarchy', fontsize=11)
ax.set_title('PoA vs. Nivel de demand')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f'))

# Panel derecho: costs UE y SO
ax2 = axes[1]
d_veh = scales * total_demand
ax2.plot(d_veh, np.array(cost_ue_curve) / 3600, 's-', color='#e74c3c',
         linewidth=2, markersize=7, label='Equilibrio Wardrop (UE)')
ax2.plot(d_veh, np.array(cost_so_curve) / 3600, '^-', color='#2ecc71',
         linewidth=2, markersize=7, label='Social optimum (SO)')
ax2.fill_between(d_veh,
                 np.array(cost_so_curve) / 3600,
                 np.array(cost_ue_curve) / 3600,
                 alpha=0.15, color='#e74c3c', label='Cost of anarchy')
ax2.set_xlabel('Demand total (veh/h)', fontsize=11)
ax2.set_ylabel('Total cost (veh·h)', fontsize=11)
ax2.set_title('Costo del sistema: UE vs SO')
ax2.legend(fontsize=9)

plt.suptitle('Price of Anarchy — Chapinero, Bogota', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'poa_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Decomposition by O-D pair

The following calculation attributes route costs to O-D pairs using reconstructed route flows. This is a diagnostic decomposition and depends on the restricted route set.


In [ ]:
# Reconstruct times UE y SO por par O-D usando times de edge
t_ue_arr = res['t_ue']
t_so_arr = res['t_so']

poa_od = []
for r_idx, ((o, d), path) in enumerate(route_list):
    pass   # usamos flows de route

# For each O-D pair: flow-weighted average time in UE and SO
f_ue_val = None

# Reconstruct f_ue by solving UE again to obtain route values
R_loc = delta.shape[1]
f_ue_var = cp.Variable(R_loc, nonneg=True)
x_expr   = delta @ f_ue_var
alpha_v, beta_v = 0.15, 4
beck = (cp.sum(cp.multiply(t0, x_expr)) +
        cp.sum(cp.multiply(t0 * alpha_v / (beta_v+1) / (cap**beta_v),
                           cp.power(x_expr, beta_v+1))))
cons_ue = []
for (o, d, dem) in od_pairs:
    if (o, d) in od_route_map:
        cons_ue.append(cp.sum(f_ue_var[od_route_map[(o, d)]]) == dem)
cp.Problem(cp.Minimize(beck), cons_ue).solve(solver=cp.CLARABEL, verbose=False)
f_ue_val = f_ue_var.value

f_so_var = cp.Variable(R_loc, nonneg=True)
x_expr2  = delta @ f_so_var
so_obj   = (cp.sum(cp.multiply(t0, x_expr2)) +
            cp.sum(cp.multiply(t0 * alpha_v / (cap**beta_v),
                               cp.power(x_expr2, beta_v+1))))
cons_so2 = []
for (o, d, dem) in od_pairs:
    if (o, d) in od_route_map:
        cons_so2.append(cp.sum(f_so_var[od_route_map[(o, d)]]) == dem)
cp.Problem(cp.Minimize(so_obj), cons_so2).solve(solver=cp.CLARABEL, verbose=False)
f_so_val = f_so_var.value

print('Flows por route reconstruidos.')

In [ ]:
rows = []
for (o, d, dem) in od_pairs:
    od_key = (o, d)
    if od_key not in od_route_map:
        continue
    idxs = od_route_map[od_key]

    # O-D cost = sum of route_flow * route_time using equilibrium edge times
    def route_cost(r_idx, t_arr):
        _, path = route_list[r_idx]
        total = 0
        edge_idx_map = {e: i for i, e in enumerate(edges_list)}
        for i in range(len(path)-1):
            u, v = path[i], path[i+1]
            if G.has_edge(u, v):
                best_k = min(G[u][v], key=lambda k: G[u][v][k].get('t0', float('inf')))
                e = (u, v, best_k)
                if e in edge_idx_map:
                    total += t_arr[edge_idx_map[e]]
        return total

    cost_od_ue = sum(f_ue_val[r] * route_cost(r, t_ue_arr) for r in idxs)
    cost_od_so = sum(f_so_val[r] * route_cost(r, t_so_arr) for r in idxs)

    poa_od_val = cost_od_ue / cost_od_so if cost_od_so > 0 else np.nan
    rows.append({
        'origin':  str(o),
        'destination': str(d),
        'demand': dem,
        'cost_ue': cost_od_ue,
        'cost_so': cost_od_so,
        'poa':     poa_od_val,
        'saving_s': (cost_od_ue - cost_od_so) / dem if dem > 0 else 0,
    })

df_poa = pd.DataFrame(rows).sort_values('poa', ascending=False)
print('PoA by O-D pair:')
display(df_poa.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
labels = [f"{r['origin'][:5]}→{r['destination'][:5]}" for _, r in df_poa.iterrows()]
colors = ['#e74c3c' if p > 1.01 else '#3498db' for p in df_poa['poa']]
bars = ax.bar(range(len(df_poa)), df_poa['poa'], color=colors, edgecolor='white', linewidth=0.4)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
ax.axhline(theoretical_bound, color='gray', linestyle=':', linewidth=1, label=f'Bound ({theoretical_bound:.2f})')
ax.set_xticks(range(len(df_poa)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Price of Anarchy', fontsize=11)
ax.set_title('PoA desagregado por par O-D — Chapinero')
ax.legend()
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'poa_by_od.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Decomposition by road type

The aggregation below groups edge-level quantities by the OSM road-class attribute. These categories are inherited from the source graph and are not independently validated here.


In [ ]:
highway_types = []
for u, v, k in edges_list:
    hw = G[u][v][k].get('highway', 'unclassified')
    if isinstance(hw, list):
        hw = hw[0]
    highway_types.append(str(hw))

df_hw = pd.DataFrame({
    'highway': highway_types,
    'x_ue':    x_ue,
    'x_so':    x_so,
    't_ue':    t_ue_arr,
    't_so':    t_so_arr,
    'cap':     cap,
})

df_hw['cost_ue'] = df_hw['x_ue'] * df_hw['t_ue']
df_hw['cost_so'] = df_hw['x_so'] * df_hw['t_so']
df_hw['vcr_ue']  = df_hw['x_ue'] / df_hw['cap']
df_hw['vcr_so']  = df_hw['x_so'] / df_hw['cap']

agg = df_hw.groupby('highway').agg(
    edges=('x_ue', 'count'),
    cost_ue=('cost_ue', 'sum'),
    cost_so=('cost_so', 'sum'),
    vcr_ue_mean=('vcr_ue', 'mean'),
    vcr_so_mean=('vcr_so', 'mean'),
).reset_index()

agg['poa_local'] = agg['cost_ue'] / agg['cost_so'].replace(0, np.nan)
agg = agg.sort_values('poa_local', ascending=False)

print('PoA local por road type:')
display(agg.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
valid = agg.dropna(subset=['poa_local'])
bars = ax.barh(valid['highway'], valid['poa_local'] - 1,
               color='#e67e22', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Exceso de PoA sobre 1 (PoA − 1)', fontsize=11)
ax.set_title('Ineficiencia por road type — Chapinero')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'poa_by_highway.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Edge-level cost ratio

For each edge with positive denominator, the local ratio is computed as

$$\frac{x_e^{UE}t_e(x_e^{UE})}{x_e^{SO}t_e(x_e^{SO})}.$$

This is an edge-level diagnostic. It is not itself a network-level Price of Anarchy.


In [ ]:
cost_a_ue = x_ue * t_ue_arr
cost_a_so = x_so * t_so_arr
poa_local = np.where(cost_a_so > 0.1, cost_a_ue / cost_a_so, 1.0)

for i, (u, v, k) in enumerate(edges_list):
    G[u][v][k]['poa_local'] = float(poa_local[i])

edge_colors = ox.plot.get_edge_colors_by_attr(
    G, attr='poa_local', cmap='YlOrRd',
    num_bins=10, na_color='#2a2a4a'
)

fig, ax = ox.plot_graph(
    G,
    edge_color=edge_colors,
    edge_linewidth=1.3,
    node_size=4,
    node_color='white',
    bgcolor='#1a1a2e',
    figsize=(12, 10),
    show=False,
    close=False,
)
ax.set_title('Price of Anarchy local by edge\n(yellow=efficient, red=inefficient) — Chapinero',
             color='white', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'poa_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary quantities

The printed values summarize the numerical instance. They should be checked against solver status and feasibility before being cited.


In [ ]:
least_efficient_pair  = df_poa.iloc[0]
most_efficient_pair = df_poa.iloc[-1]

print('=' * 60)
print('   SUMMARY - PRICE OF ANARCHY, CHAPINERO BOGOTA')
print('=' * 60)
print(f'  PoA global            : {poa:.5f}  (+{(poa-1)*100:.3f}%)')
print(f'  Theoretical bound Roughgarden: {theoretical_bound:.4f}  (+{(theoretical_bound-1)*100:.1f}%)')
print(f'  % of bound reached: {poa/theoretical_bound*100:.1f}%')
print()
print(f'  Least efficient O-D pair : {least_efficient_pair["origin"][:8]}→{least_efficient_pair["destination"][:8]}')
print(f'    PoA = {least_efficient_pair["poa"]:.4f}  |  potential saving = {least_efficient_pair["saving_s"]:.1f} s/veh')
print()
print(f'  Most efficient O-D pair   : {most_efficient_pair["origin"][:8]}→{most_efficient_pair["destination"][:8]}')
print(f'    PoA = {most_efficient_pair["poa"]:.4f}')
print()
print(f'  Interpretation:')
print(f'  The decentralized equilibrium costs {(poa-1)*100:.2f}% more than')
print(f'  el social optimum. Con 2x demand, el PoA sube a ~{poa_curve[-1]:.4f}.')
print('=' * 60)
print('\nReady for Notebook 05 - Signal timing (Webster)')

## 8. Save intermediate results

The saved object is used by the signal-timing notebook.

Limitations of this notebook:
- PoA values depend on the finite route set and synthetic demands.
- Sensitivity calculations may fail or return inaccurate solver statuses.
- Edge-level ratios are diagnostic quantities, not separate definitions of network PoA.


In [ ]:
results_poa = {
    **res,
    'poa':          poa,
    'theoretical_bound': theoretical_bound,
    'poa_curve':    poa_curve,
    'scales':       scales.tolist(),
    'df_poa':       df_poa,
    'poa_local':    poa_local,
    'cost_a_ue':    cost_a_ue,
    'cost_a_so':    cost_a_so,
}

out_path = PROCESSED_DIR / 'results_poa.pkl'
with open(out_path, 'wb') as fh:
    pickle.dump(results_poa, fh)

print(f'Resultados PoA guardados en: {out_path}')

In [ ]:
print(type(f_ue_val), f_ue_val is None)
print(type(f_so_val), f_so_val is None)

In [ ]:
import subprocess
result = subprocess.run(['grep', '-n', 'CLARABEL', 
    '/home/nicolasmelendez30/signals-bogota/notebooks/04_price_of_anarchy.ipynb'],
    capture_output=True, text=True)
print(result.stdout)
